# Transferring Topo stuff to python

In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti
from myPythonTools.TopoAna import rdremap as RM
from myPythonTools.TopoAna import rdridgelist as RL

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)
importlib.reload(RM)
importlib.reload(RL)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
from scipy.ndimage import uniform_filter
def smooth_gauss_scale_log(gs, n=5):
    """
    Geometric-mean box smoothing of gauss_scale: exp( box-mean( log(gs) ) ).

    gs : array (6, nc, nc) as reshaped from the topo_smooth file, or one face (nc, nc).
    n  : full box width in cells (3, 5, 7, ...). Even n is rounded up; n < 3 returns a copy.

    Same rules as smooth_gauss_scale_log in smooth_topo_cube.F90:
      - zeros mean "never ridge-like": they are left out of the average,
        and a zero cell stays zero (so the excluded-seed set is unchanged);
      - each face is smoothed on its own, with the edge value repeated
        (no exchange across face edges).
    """
    gs = np.asarray(gs, dtype=np.float64)
    if n < 3:
        return gs.copy()
    n = n + 1 if n % 2 == 0 else n
    size = (n, n) if gs.ndim == 2 else (1, n, n)       # never smooth across faces

    valid = gs > 0.0
    logs = np.where(valid, np.log(np.where(valid, gs, 1.0)), 0.0)

    # box means of (log * valid) and of valid; their ratio is the mean log over valid cells
    sum_log = uniform_filter(logs, size=size, mode="nearest")
    count = uniform_filter(valid.astype(np.float64), size=size, mode="nearest")

    out = np.zeros_like(gs)
    ok = valid & (count > 0.0)                         # a valid cell always counts itself
    out[ok] = np.exp(sum_log[ok] / count[ok])
    return out


# Example:
#   gs   = ds['gauss_scale'].values.reshape(6, nc, nc)
#   gs5  = smooth_gauss_scale_log(gs, 5)
#   nswx = np.clip(np.rint(c * gs5 / 3.3356), 4, 42)     # what --adaptive_window would use

In [ ]:

ogrid= 'ne120pg3'
smoothing_factor=25
tag='X02'

ogrid= 'ne120pg3'
smoothing_factor=100
tag='XtraSmooth'

ogrid= 'ne30pg3'
smoothing_factor=100
tag='Rw050_v204_ndguard_e01'
tag='Rw050_v205_new_wedge_e01'

TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'

Case = f'{ogrid}_Sco{str(smoothing_factor)}_{tag}'

#Case = 'ne30pg3_Sco100_Rw050_It02_Wm1_v206_e01'
#Case = 'ne30pg3_Sco100_Rw050_Wm1_v206_e02'
#Case ='ne30pg3_Sco100_Aw50_Wm1_adptv_window_v300_e01'
#Case ='ne30pg3_Sco100_It02_Aw50_Wm1_adptv_window_v300_e02'
#Case ='ne30pg3_Sco100_It02_Aw50_Wm1_adptv_window_v301_e01'
#Case ='ne30pg3_Sco100_It03_Aw50_Wm1_adptv_window_v301_e02'
#Case ='ne30pg3_Sco100_It03_Aw50_Wm1_adptv_window_v302_e03'
Case ='ne30pg3_Sco100_It03_Aw50_Wm1_Yn00_adptv_window_v303_e01'
Case ='ne30pg3_Sco100_It02_Aw50_adptv_window_v302_e02'
Case ='ne30pg3_Sco100_It02_Aw50_Wm1_Yn00_Px20_Gs05_adptv_window_v304_e02'
#Case ='ne30pg3_Sco100_v202_rch1p0_e01'
print(Case)

In [ ]:
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/remap*dat')
RmpFile = foo[-1]
print(RmpFile)
Rmp=RM.rdremap( RmpFile,  fields="all" )

foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo*.nc')
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )

if 'gauss_scale' in list(topo.variables):
    gauss_scale = np.reshape( topo['gauss_scale'].values , (6,nc,nc) )
    gauss_strng = np.reshape( topo['gauss_strength'].values , (6,nc,nc) )
    gauss_angle = np.reshape( topo['gauss_angle'].values , (6,nc,nc) )
else:
    gauss_scale = np.zeros( (6,nc,nc) )
    gauss_strng = np.zeros( (6,nc,nc) )
    gauss_angle = np.zeros( (6,nc,nc) )

gauss_scale_sm=smooth_gauss_scale_log(gauss_scale, n=5)

# Define the range of indices
x_indices = np.arange(3000)
y_indices = np.arange(3000)

# Use meshgrid to create 2D arrays of x and y indices
xgrid, ygrid = np.meshgrid(x_indices, y_indices, indexing='xy')


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
ListFile = foo[-1]
List=RL.rdridgelist( ListFile )


In [ ]:
print(List.keys())
print(List.rdg_profiles.shape)
print(List.rdg_profiles_x.shape)



In [ ]:
print(Rmp.keys())


wedge=Rmp.wedge
wedgo=Rmp.wedgo
nodos=Rmp.nodos
mxdis=Rmp.mxdis
profi=Rmp.profi
uniqid=Rmp.uniqid

xspk=List.xspk
yspk=List.yspk
xs=List.xs
ys=List.ys
panl=List.MyPanel

In [ ]:
ipan,SWyx,wlen =3,[2850,950],50
xlims=np.asarray( (0,wlen) )+SWyx[1]
ylims=np.asarray( (0,wlen) )+SWyx[0]

plt.contourf( dev[3,:,:] , levels=np.linspace(-500,1000,num=21), cmap='terrain')
plt.xlim( xlims )
plt.ylim( ylims )


In [ ]:
ipan=3
#plt.plot( uniqid[ipan,2610:2630,640:670 ].flatten() ,'.' )
#plt.ylim(183200,184000)

###################################################
# Set region of interest
# Baja
ipan,SWyx,wlen =3,[2500,575],200
xlims=np.asarray( (0,wlen) )+SWyx[1]
ylims=np.asarray( (0,wlen) )+SWyx[0]

# CO Front Range
ipan,SWyx,wlen =3,[2850,950],50
xlims=np.asarray( (0,wlen) )+SWyx[1]
ylims=np.asarray( (0,wlen) )+SWyx[0]


##################################################

uoo=uniqid[ipan, ylims[0]:ylims[1], xlims[0]:xlims[1] ].flatten()
mxdoo=mxdis[ipan, ylims[0]:ylims[1], xlims[0]:xlims[1] ].flatten()
uoo=uoo[  (uoo>0)&(mxdoo>700.)  ]
uoo=(np.unique(uoo) )
iuoo=uoo.astype(int)
print(iuoo)
print( List.hwdth[iuoo-1])
print( List.mxdis[iuoo-1])

quoo=iuoo

xuoo=List.xspk[quoo-1]-1
yuoo=List.yspk[quoo-1]-1
xsoo=List.xs[quoo-1]-1
ysoo=List.ys[quoo-1]-1
puoo=List.MyPanel[quoo-1]-1

#xuoo=xuoo.astype(int)
#yuoo=yuoo.astype(int)
#puoo=puoo.astype(int)

print( List.istop_rdg[quoo] )
print( List.sigwin_rdg[iuoo[-1]-1] )

In [ ]:
inj=1
worst=0
for ir in iuoo[inj:inj+1]: 
    xbig = np.arange( 0, len(List.rdg_profiles_x[ir-1,:] ) )
    nswx = List.NSWx_diag[ir-1]
    nswB = len(List.rdg_profiles_x[ir-1,:] ) 
    shift_x = (nswB-nswx-1)/2. # - 1
    xsmall = np.arange( 0, nswx+1 ) + shift_x
    plt.plot( xbig, List.rdg_profiles_x[ir-1,:]  )
    plt.plot( xsmall, List.rdg_profiles[ir-1,:nswx+1] , '--' )
    plt.plot( List.xnodes_list[ir-1,:] -1 + shift_x ,List.hnodes_list[ir-1,:] ,'.' )
    plt.plot( List.xwedge_list[ir-1,:] -1 + shift_x ,List.hwedge_list[ir-1,:] ,'o' )
    print( List.mxdis[ir-1],List.NSWx_diag[ir-1], len(List.rdg_profiles[ir-1,:] ) )
    print( List.sigwin_rdg[ir-1],List.anglx[ir-1] )

    ipk=ir-1
    n  = int(List.NSWx_diag[ipk])
    a  = List.rdg_profiles[ipk, :n+1]                 # ANISO_ANA_2, wedge profile
    i0 = List.PSW - (n - n//2)                        # same centre offset, 0-based
    b  = List.rdg_profiles_x[ipk, i0:i0+n+1]          # ANISO_ANA_3, middle part
    worst = max(worst, np.max(np.abs(a - b)))
    plt.plot( a )
    plt.plot( b, '+' )
    print( a-b, worst )


In [ ]:
print(List.keys())
print(List.rdg_profiles.shape)
plt.plot( List.rdg_profiles[iuoo[-1]-1,:] )
plt.plot( List.rdg_profiles[iuoo[0]-1,:] )
plt.plot( List.rdg_profiles[iuoo[1]-1,:] )
plt.plot( List.rdg_profiles[iuoo[2]-1,:] )


In [ ]:
#plt.figure(figsize=(20, 20))
clevels=np.linspace(-500.,3500.,num=21)
gslevels=np.linspace(0.,1000.,num=21)
dlevels=np.linspace(-2000.,2000.,num=21)
wlevels=np.linspace(0,100,num=21)
cmap=plt.cm.terrain

# Deferring to region of interest chosen above
"""
#idx_ridge = 183_559
idx_ridges = np.asarray( [183_559,183_381] )


ipan,SWyx,wlen =3,[2000,100],1000
ipan,SWyx,wlen =3,[2500,1300],500
ipan,SWyx,wlen =0,[2000,1000],1000
ipan,SWyx,wlen =3,[2500,575],200

xlims=np.asarray( (0,wlen) )+SWyx[1]
ylims=np.asarray( (0,wlen) )+SWyx[0]

# Baja ...
#ipan=3
#xlims=(600,700) #( 200,1200 )
#ylims=(2600,2700) #( 2000, 3000 )

xoo = xs[panl==ipan+1]-1
yoo = ys[panl==ipan+1]-1

"""

fig,axs = plt.subplots(2,4,figsize=(60,2*13) )
axs=axs.flatten()

i=0
ax=axs[i]
c0=ax.contourf( dev[ipan,:,:]  ,levels=clevels,cmap=cmap , extend='both' )
ax.scatter( xuoo,yuoo,marker='x', s=200, color='red')
ax.scatter( xsoo,ysoo,marker='+', s=200, color='red')
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "Topo Dev" )
cb0=plt.colorbar(c0,ax=ax,shrink=0.7 )

i=i+1
ax=axs[i]
c0=ax.contourf( gauss_scale_sm[ipan,:,:] ,levels=wlevels,cmap=cmap , extend='both' )
#c2=ax[1].imshow( mxdis[ipan,:,:] ,min=-500,max=3500,cmap=cmap , extend='both' )
#ax.scatter( xs[idx_ridges]-1,ys[idx_ridges]-1,marker='+', s=200, color='red')
#ax.scatter( xspk[idx_ridges]-1,yspk[idx_ridges]-1,marker='x', s=200, color='red')
#ax.scatter( xspk[iuoo[-1]-1]-1,yspk[iuoo[-1]-1]-1,marker='x', s=200, color='red')
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "Gauss scale" )
cb=plt.colorbar(c0,ax=ax,shrink=0.7 )

i=i+1
ax=axs[i]
#c0=ax.contourf( gauss_scale[ipan,:,:] ,levels=wlevels,cmap=cmap , extend='both' )
c0=ax.contourf( gauss_strng[ipan,:,:] ,levels=gslevels,cmap=cmap , extend='both' )
#c2=ax[1].imshow( mxdis[ipan,:,:] ,min=-500,max=3500,cmap=cmap , extend='both' )
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "Gauss strength" )
cb=plt.colorbar(c0,ax=ax,shrink=0.7 )

i=i+1
ax=axs[i]
c0=ax.contourf( mxdis[ipan,:,:] ,levels=clevels,cmap=cmap , extend='both' )
#c2=ax[1].imshow( mxdis[ipan,:,:] ,min=-500,max=3500,cmap=cmap , extend='both' )
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "mxdis" )
cb=plt.colorbar(c0,ax=ax,shrink=0.7 )

i=i+1
ax=axs[i]
c0=ax.contourf( wedgo[ipan,:,:] ,levels=clevels,cmap=cmap , extend='both' )
#c2=ax[1].imshow( mxdis[ipan,:,:] ,min=-500,max=3500,cmap=cmap , extend='both' )
#ax.scatter( xspk[iuoo[-1]-1]-1,yspk[iuoo[-1]-1]-1,marker='x', s=200, color='red')
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "wedgo" )
cb=plt.colorbar(c0,ax=ax,shrink=0.7 )

i=i+1
ax=axs[i]
c0=ax.contourf( nodos[ipan,:,:] ,levels=clevels,cmap=cmap , extend='both' )
#c2=ax[1].imshow( mxdis[ipan,:,:] ,min=-500,max=3500,cmap=cmap , extend='both' )
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "nodos" )
cb=plt.colorbar(c0,ax=ax,shrink=0.7 )

i=i+1
ax=axs[i]
c0=ax.contourf( profi[ipan,:,:] ,levels=clevels,cmap=cmap , extend='both' )
#c2=ax[1].imshow( mxdis[ipan,:,:] ,min=-500,max=3500,cmap=cmap , extend='both' )
ax.set_xlim( xlims )
ax.set_ylim( ylims )
ax.set_aspect('equal') #,'box')
ax.set_title( "profi" )
cb=plt.colorbar(c0,ax=ax,shrink=0.7 )


fig.suptitle(
    f"{Case}",
    fontsize=15,
    y=0.90
)


In [ ]:
print( poopypants)

In [ ]:
print(ListFile)

In [ ]:
import numpy as np
from rdridgelist import rdridgelist

r = rdridgelist(ListFile, profiles=True)
ok = (r.mxdis > 0) & (r.NSWx_diag > 0)
worst = 0.0 * r.mxdis
for ipk in np.nonzero(ok)[0]:
    #for ipk in [130_028,]:
    n  = int(r.NSWx_diag[ipk])
    a  = r.rdg_profiles[ipk, :n+1]                 # ANISO_ANA_2, wedge profile
    i0 = r.PSW - (n - n//2)                        # same centre offset, 0-based
    b  = r.rdg_profiles_x[ipk, i0:i0+n+1]          # ANISO_ANA_3, middle part
    worst[ipk] = max(worst[ipk], np.max(np.abs(a - b)))
#print("largest difference (m):", worst)

In [ ]:
ipk=130_028
n  = int(r.NSWx_diag[ipk])
a  = r.rdg_profiles[ipk, :n+1]                 # ANISO_ANA_2, wedge profile
i0 = r.PSW - (n - n//2)                        # same centre offset, 0-based
b  = r.rdg_profiles_x[ipk, i0:i0+n+1]          # ANISO_ANA_3, middle part
plt.plot( r.rdg_profiles[ipk, :n+1]  )
plt.plot( r.rdg_profiles_x[ipk,i0:i0+n+1]   )

In [ ]:
plt.plot( worst , '.')

gloob = np.sum( worst > 0 )
print( gloob )
#plt.xlim( 130_020,130_030)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

diff=worst
ok   = (r.mxdis > 0) & (r.NSWx_diag > 0) & (r.lonc > -9000.)   # -9999 = no crest point
lon  = np.degrees(r.lonc)                                     # lonc/latc are in RADIANS
lat  = np.degrees(r.latc)
lon  = (lon + 180.) % 360. - 180.
bad  = ok & (diff > 1.0)
good = ok & ~bad

fig = plt.figure(figsize=(16, 8))
ax  = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(lw=0.4)
ax.scatter(lon[good], lat[good], s=0.1, c='0.8', transform=ccrs.PlateCarree())
sc = ax.scatter(lon[bad], lat[bad], s=4, c=diff[bad], cmap='viridis',
                vmin=0, vmax=np.percentile(diff[bad], 95), transform=ccrs.PlateCarree())

# cube face edges, from the topo file's (6, nc, nc) cell-centre lon/lat
for f in range(6):
    for edge in (np.s_[f, 0, :], np.s_[f, -1, :], np.s_[f, :, 0], np.s_[f, :, -1]):
        ax.plot(tlons[edge], tlats[edge], 'r-', lw=0.6, transform=ccrs.Geodetic())

plt.colorbar(sc, shrink=0.6, label='largest profile difference (m)')
ax.set_title(f"{bad.sum()} of {ok.sum()} ridgelets disagree")
plt.show()

In [ ]:
xspk.shape


In [ ]:
print( xspk[idx_ridges]-1,yspk[idx_ridges]-1 )

In [ ]:
uiooo = uniqid[(uniqid>0)]-1


In [ ]:
plt.scatter( xoo,yoo,marker='+')

In [ ]:
error=dev-profi
plt.plot( error[ipan,:,:].flatten() )
print( error[np.abs(error)>10.].std() )



In [ ]:
errorH=np.histogram( np.abs(error), range=(10,3000) , bins=50 )
plt.plot( errorH[1][1:] , np.log(errorH[0]+1) , '-o')

In [ ]:
# Do a file search in output directory
# Pick last (most recent) file
fpat = TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat'
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
print( fpat )
ListFile = foo[-1]
List=RL.rdridgelist( ListFile )

In [ ]:
fig,ax=plt.subplots( 2,2, figsize=(16,12) )

ax=ax.flatten()
dlevs = np.linspace( 0,2000, num=21 )
wlevs = np.linspace( 0,1, num=11 )

cmap='inferno'

c0=ax[0].tricontourf( lonc, latc, dCAM.ISOVAR.values ,levels=dlevs, cmap=cmap )
c1=ax[1].tricontourf( lonc, latc, dCAM.ISOWGT.values ,levels=wlevs, cmap=cmap )
c2=ax[2].tricontourf( lonc, latc, dCAM.SGH.values ,levels=dlevs, cmap=cmap  )
c3=ax[3].tricontourf( lonc, latc, dCAM.MXDIS[0,:].values ,levels=dlevs, cmap=cmap  )

plt.colorbar(c0 )
plt.colorbar(c1 )
plt.colorbar(c2 )
plt.colorbar(c3 )



In [ ]:
Po.sixpanel(aa=raw, clev=np.linspace(0,5000.,num=21) ,goofy=True , cmap='terrain' )
#Po.sixpanel(aa=yoog, clev=np.linspace(-1,2,num=21) )

In [ ]:
Po.sixpanel(aa=nodos, clev=np.linspace(-500,3000.,num=21) ,goofy=True , cmap='terrain' )
#Po.sixpanel(aa=yoog, clev=np.linspace(-1,2,num=21) )

In [ ]:
#plt.plot( dev[3,2800,:])
residual = dev - nodos #wedge


In [ ]:
ipanel=3

xlim=(500,1000) #1000
ylim=(2500,3000)
#xlim=(0,3000)
#ylim=(0,3000)

clevels=np.linspace(-500.,3500.,num=21)
cmap=plt.cm.terrain
fig,ax = plt.subplots(2,2,figsize=(15,15) ,squeeze=False ) # , sharex=True)

titlesize=16


i,j=0,1
cf=ax[i,j].contourf( dev[ipanel,:,:],cmap=cmap,levels=clevels)
#cf=ax[0,0].contour( mxdis[ipanel,:,:],colors='black',levels=clevels)
#cf=ax[0,0].contourf( raw[ipanel,:,:],cmap=cmap,levels=clevels) #,alpha=.99)
#ax[0].contour( tlons[4,:,:], colors='white',clevels=51)
#plt.colorbar(cf,ax=ax[0,0],location='bottom')
ax[i,j].set_aspect('equal') #,'box')
ax[i,j].set_xlim( xlim )
ax[i,j].set_ylim( ylim )
ax[i,j].set_title( "Topographic deviation from smoothed Topo for bottom BC" , fontsize=titlesize )

#ax[1].contourf( block[4,:,:])
#ax[1].set_aspect('equal') #,'box')


i,j=1,1
"""
cf=ax[i,j].contourf( wedgo[ipanel,:,:],cmap=cmap,levels=clevels)
ax[i,j].set_aspect('equal') #,'box')
ax[i,j].set_xlim( xlim )
ax[i,j].set_ylim( ylim )
ax[i,j].set_title( "Volume filling reconstruction from ridge parameters" , fontsize=titlesize )
"""
cf=ax[i,j].contourf( mxdis[ipanel,:,:],cmap=cmap,levels=clevels)
ax[i,j].set_aspect('equal') #,'box')
#ax[i,j].set_xlim( (2200,2700) )
#ax[i,j].set_ylim( (500,1000) )
ax[i,j].set_ylim( (0,1000) )
ax[i,j].set_xlim( left=0,right=3000 , auto=True)
ax[i,j].set_title( "Ridge-line skeleton" , fontsize=titlesize )


i,j=0,0
cf=ax[i,j].contourf( raw[ipanel,:,:],cmap=cmap,levels=clevels)
ax[i,j].set_aspect('equal') #,'box')
ax[i,j].set_xlim( xlim )
ax[i,j].set_ylim( ylim )
ax[i,j].set_title( "Raw 3km X 3km Topography" , fontsize=titlesize )

i,j=1,0
cf=ax[i,j].contourf( mxdis[ipanel,:,:],cmap=cmap,levels=clevels)
ax[i,j].set_xlim( xlim )
ax[i,j].set_ylim( ylim )
ax[i,j].set_aspect('equal') #,'box')
ax[i,j].set_title( "Ridge-line skeleton" , fontsize=titlesize )

i,j=1,1
ipanel=0
cf=ax[i,j].contourf( mxdis[ipanel,:,:],cmap=cmap,levels=clevels)
#ax[i,j].set_xlim( (2200,2700) )
#ax[i,j].set_ylim( (500,1000) )
ax[i,j].set_xlim( (0,3000) )
ax[i,j].set_ylim( (0,3000) )
ax[i,j].set_aspect('equal') #,'box')
#ax[i,j].set_ylim( (0,1000) )
#ax[i,j].set_xlim( left=0,right=3000 , auto=True)
ax[i,j].set_title( "Ridge-line skeleton" , fontsize=titlesize )


plt.tight_layout()

plt.colorbar(cf,ax=ax,location='bottom',shrink=.4)



## Top-left shows raw 3km X 3km topography. For topographic BC generation, a smoothing scale is chosen first.  For 1 degree grids topography is typically smoothed over a ~180km radius. Top-right shows "deviation" of raw topography from topography smoothed with a 180km radius. Bottom-left shows "skeleton" derived by analyzing varaiances in rotated boxes with 180/sqrt(2) sides. Ridge-orientation is defined as the rotation angle with maximum variance of ridge profile (Y-average of topography, where Y is the coordinate along the potential ridge line).  Bottom-left shows a volume-filling reconstruction of the topography based on ridge parameters - heght, orientation, width, length

In [ ]:
ipanel=3

xlim=(300,1900)
ylim=(1400,3000)

clevels=np.linspace(-500.,3500.,num=21)
cmap=plt.cm.terrain
fig,ax = plt.subplots(1,1,figsize=(15,15) ,squeeze=False , sharex=True)

titlesize=16


i,j=0,0
cf=ax[i,j].contour( raw[ipanel,:,:],colors='white',levels=[1,10000])
cf=ax[i,j].contour( mxdis[ipanel,:,:],colors='black',levels=clevels)
cf=ax[i,j].contourf( dev[ipanel,:,:],cmap=cmap,levels=clevels)
#cf=ax[0,0].contourf( raw[ipanel,:,:],cmap=cmap,levels=clevels) #,alpha=.99)
#ax[0].contour( tlons[4,:,:], colors='white',clevels=51)
#plt.colorbar(cf,ax=ax[0,0],location='bottom')
ax[i,j].set_aspect('equal') #,'box')
ax[i,j].set_xlim( xlim )
ax[i,j].set_ylim( ylim )
ax[i,j].set_title( "Topographic deviation with skeleton over-plotted" , fontsize=titlesize )

ax[i,j].annotate("Nevada",
            xy=(700, 2900), xycoords='data',
            xytext=(1200, 2700), textcoords='data',color='white',
            arrowprops=dict(arrowstyle="->",
                            connectionstyle="arc3",color='white'),
            )
#ax[1].contourf( block[4,:,:])
#ax[1].set_aspect('equal') #,'box')
plt.colorbar(cf,ax=ax,location='right',shrink=.4)

## Ridge skeleton overlaid on topographic deviation. Notice that deviation is high-pass and multi-scale, in other words, all scales smaller than the smoothing scale are retained. But, ridge algorithm tends to pick out features with scales on the order of the smoothng radius.The algorithm can pick out smaller scales only where no feature with scale~smoothing radius is present, e.g., Nevada